### Exercici 1 — Presentació del  conjunt de dades
Descripció del sector, objectiu de l’anàlisi i 
explicació de les columnes.

### Exercici 2 — Exploració inicial
Càrrega del CSV, visualització de les primeres files, 
dimensions del conjunt de dades i tipus de dades.

In [14]:
import csv
from datetime import datetime

nom_fitxer = "01_botiga_esport.csv"

with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    camps = [camp.strip() for camp in lector.fieldnames]
    registres = [
        {camp.strip() if camp is not None else None: valor for camp, valor in reg.items()}
        for reg in lector
    ]

print("Camps del CSV:", camps)
print("Nombre total de registres:", len(registres))
print("Exemple de primer registre:", registres[0])


Camps del CSV: ['article', 'familia_esportiva', 'botiga', 'unitats', 'distancia_km', 'data']
Nombre total de registres: 120
Exemple de primer registre: {'article': 'Pantalons curts  ', 'familia_esportiva': ' Material de raqueta', 'botiga': '          ', 'unitats': ' 1O        ', 'distancia_km': ' 97.6        ', 'data': ' 2025-08-27'}


### Exercici 3 — Detecció d'incidències
identificació de camps obligatoris buits, valors numèrics 
incorrectes, dates invàlides i inconsistències de format. 

In [ ]:
registres_incomplets = 0
valors_numerics_incorrectes = 0
dates_invalides = 0
formats_incorrectes = 0

for reg in registres:
    # comprobar vacio
    te_buit = any(val is None or str(val).strip() == "" for val in reg.values())
    #valor incorrecto
    valor_incorrecto = any(not str(reg[camp]).strip().replace(".", "", 1).replace(",", "", 1).isdigit()
        for camp in ["unitats", "distancia_km"]
        if reg.get(camp) is not None and str(reg[camp]).strip() != ""
    )
    #comprobar fecha
    data_invalida = False
    try:
        from datetime import datetime
        datetime.strptime(str(reg["data"]).strip(), "%Y-%m-%d")
    except (ValueError, TypeError, KeyError):
        data_invalida = True
    #comprobar formato incorrecto
    format_incorrecto = any(
        isinstance(val, str) and val != val.strip()
        for val in reg.values()
    )
    #sumar contadores
    if te_buit:
        registres_incomplets += 1
    if format_incorrecto:
        formats_incorrectes += 1
    if valor_incorrecto:
        valors_numerics_incorrectes += 1
    if data_invalida:
        dates_invalides += 1
    

print(f"Registres amb almenys un camp buit/espais: {registres_incomplets}")
print(f"Registres amb valors numèrics incorrectes: {valors_numerics_incorrectes}")
print(f"Registres amb dates invàlides: {dates_invalides}")
print(f"Registres amb formats incorrectes: {formats_incorrectes}")



### Exercici 4 — Neteja i validació
Aplicació de les transformacions necessàries i definició de les 
regles per determinar quins registres són vàlids. 

In [ ]:
vendes_netes = []
incidencies = []
rebutjades = []

obligatoris = ["article", "familia_esportiva", "botiga"]

for idx, reg in enumerate(registres, start=2):
    es_valid = True
    reg_net = {}

    # 1. Comprovació de camps obligatoris
    for camp in obligatoris:
        val_original = reg.get(camp, "")
        val_net = str(val_original or "").strip()
        if not val_net:
            es_valid = False
            incidencies.append({
                "fila_csv": idx, "tipus": "REBUTJAT", "camp": camp,
                "valor_original": val_original, "accio": "Camp obligatori buit"
            })
        else:
            if val_original != val_net:
                incidencies.append({
                    "fila_csv": idx, "tipus": "CORRECCIO", "camp": camp,
                    "valor_original": val_original, "accio": "Espais sobrants eliminats"
                })
            reg_net[camp] = val_net

    # 2. Conversió d'unitats
    unitats_original = reg.get("unitats", "")
    try:
        unitats_val = int(str(unitats_original or "").strip())
        if unitats_val <= 0:
            raise ValueError
        reg_net["unitats"] = unitats_val
    except ValueError:
        es_valid = False
        incidencies.append({
            "fila_csv": idx, "tipus": "REBUTJAT", "camp": "unitats",
            "valor_original": unitats_original, "accio": "Enter positiu no vàlid"
        })

    # 3. Conversió de distància
    distancia_original = reg.get("distancia_km", "")
    distancia_net = str(distancia_original or "").strip().strip('"')
    try:
        distancia_val = float(distancia_net)
        if not 0 <= distancia_val < float("inf"):
            raise ValueError
        reg_net["distancia_km"] = distancia_val
        if distancia_original != distancia_net:
            incidencies.append({
                "fila_csv": idx, "tipus": "CORRECCIO", "camp": "distancia_km",
                "valor_original": distancia_original, "accio": "Espais i cometes eliminats"
            })
    except ValueError:
        es_valid = False
        incidencies.append({
            "fila_csv": idx, "tipus": "REBUTJAT", "camp": "distancia_km",
            "valor_original": distancia_original, "accio": "Distància no vàlida"
        })

    # 4. Validació de data
    data_original = reg.get("data", "")
    data_neta = str(data_original or "").strip()
    try:
        datetime.strptime(data_neta, "%Y-%m-%d")
        reg_net["data"] = data_neta
        if data_original != data_neta:
            incidencies.append({
                "fila_csv": idx, "tipus": "CORRECCIO", "camp": "data",
                "valor_original": data_original, "accio": "Espais sobrants eliminats"
            })
    except ValueError:
        es_valid = False
        incidencies.append({
            "fila_csv": idx, "tipus": "REBUTJAT", "camp": "data",
            "valor_original": data_original, "accio": "Data amb format incorrecte o no vàlida"
        })

    # Classificació final del registre
    if es_valid and None not in reg:
        vendes_netes.append(reg_net)
    else:
        rebutjades.append(reg)
        if None in reg:
            incidencies.append({
                "fila_csv": idx, "tipus": "REBUTJAT", "camp": "columnes_extra",
                "valor_original": reg[None], "accio": "Valors fora de les columnes del CSV"
            })

print(f"Llegits: {len(registres)}")
print(f"Vàlids: {len(vendes_netes)}")
print(f"Rebutjats: {len(rebutjades)}")
print(f"Incidències documentades: {len(incidencies)}")


### Exercici 5 — Gestió dels registres incorrectes
Identificació dels registres rebutjats i 
explicació dels motius pels quals no es poden acceptar. 

In [20]:
# Comptar els motius pels quals s'han rebutjat registres
motius = {}
files_rebutjades = {}

for incidencia in incidencies:
    if incidencia["tipus"] == "REBUTJAT":
        motiu = incidencia["accio"]
        motius[motiu] = motius.get(motiu, 0) + 1
        fila = incidencia["fila_csv"]
        files_rebutjades.setdefault(fila, []).append(incidencia["camp"])

print(f"Total de registres rebutjats: {len(rebutjades)}")
print("Motius del rebuig:")
for motiu, quantitat in sorted(motius.items(), key=lambda element: element[1], reverse=True):
    print(f"- {motiu}: {quantitat}")

print("Files rebutjades i camps incorrectes:")
for fila, camps_incorrectes in sorted(files_rebutjades.items()):
    print(f"- Fila {fila}: {', '.join(camps_incorrectes)}")


Total de registres rebutjats: 36
Motius del rebuig:
- Data amb format incorrecte o no vàlida: 19
- Camp obligatori buit: 14
- Enter positiu no vàlid: 12
- Valors fora de les columnes del CSV: 10
- Distància no vàlida: 9
Files rebutjades i camps incorrectes:
- Fila 2: botiga, unitats
- Fila 5: data
- Fila 9: distancia_km, data
- Fila 11: familia_esportiva
- Fila 12: data, columnes_extra
- Fila 15: data
- Fila 19: familia_esportiva, unitats
- Fila 21: data, columnes_extra
- Fila 23: data
- Fila 24: data, columnes_extra
- Fila 25: article
- Fila 31: data, columnes_extra
- Fila 36: data, columnes_extra
- Fila 49: article, unitats, data, columnes_extra
- Fila 56: familia_esportiva, botiga
- Fila 61: data
- Fila 67: unitats, distancia_km
- Fila 68: data, columnes_extra
- Fila 69: article
- Fila 71: article, familia_esportiva, unitats
- Fila 72: unitats, distancia_km
- Fila 75: botiga
- Fila 77: distancia_km
- Fila 78: unitats, data, columnes_extra
- Fila 81: botiga
- Fila 88: distancia_km, d

### Exercici 6 — Anàlisi  de  les  dades  netes
càlcul d’estadístiques i indicadors rellevants, amb 
agrupacions per les variables de classificació quan sigui convenient. 

In [21]:
# GUIA DE CÀLCUL:
# 1. Sumar les unitats i calcular estadístiques de distància.
# 2. Agrupar les unitats per família esportiva i botiga.

total_unitats = sum(v["unitats"] for v in vendes_netes)
distancia_mitjana = round(
    sum(v["distancia_km"] for v in vendes_netes) / len(vendes_netes), 2
) if vendes_netes else 0.0
distancia_minima = min((v["distancia_km"] for v in vendes_netes), default=0.0)
distancia_maxima = max((v["distancia_km"] for v in vendes_netes), default=0.0)

unitats_per_familia = {}
unitats_per_botiga = {}

for v in vendes_netes:
    familia = v["familia_esportiva"]
    botiga = v["botiga"]
    unitats_per_familia[familia] = unitats_per_familia.get(familia, 0) + v["unitats"]
    unitats_per_botiga[botiga] = unitats_per_botiga.get(botiga, 0) + v["unitats"]

print(f"Total d'unitats: {total_unitats}")
print(f"Distància mitjana: {distancia_mitjana} km")
print(f"Distància mínima: {distancia_minima} km")
print(f"Distància màxima: {distancia_maxima} km")
print("Unitats per família esportiva:", unitats_per_familia)
print("Unitats per botiga:", unitats_per_botiga)


Total d'unitats: 3387
Distància mitjana: 126.78 km
Distància mínima: 2.8 km
Distància màxima: 238.9 km
Unitats per família esportiva: {'Accessoris': 446, 'Material de raqueta': 678, 'Roba': 666, 'Pilotes': 818, 'Calçat': 779}
Unitats per botiga: {'Lleida': 485, 'Manresa': 822, 'Barcelona': 537, 'Girona': 892, 'Tarragona': 651}


### Exercici 7 — Resultats i conclusions
Resum dels principals resultats i valoració de la qualitat 
de les dades.

In [ ]:
# Interpretar els resultats principals
top_familia = max(unitats_per_familia.items(), key=lambda x: x[1]) if unitats_per_familia else (None, 0)
top_botiga = max(unitats_per_botiga.items(), key=lambda x: x[1]) if unitats_per_botiga else (None, 0)

# Trobar l'article amb més unitats venudes
unitats_per_article = {}
for v in vendes_netes:
    article = v["article"]
    unitats_per_article[article] = unitats_per_article.get(article, 0) + v["unitats"]
top_article = max(unitats_per_article.items(), key=lambda x: x[1]) if unitats_per_article else (None, 0)

percentatge_valids = round(len(vendes_netes) / len(registres) * 100, 1) if registres else 0
motiu_principal = max(motius.items(), key=lambda x: x[1]) if motius else (None, 0)

print(f"Registres vàlids: {len(vendes_netes)} de {len(registres)}")
print(f"Percentatge de registres vàlids: {percentatge_valids}%")
print(f"Total d'unitats: {total_unitats}")
print(f"Família amb més unitats: {top_familia[0]} ({top_familia[1]} unitats)")
print(f"Botiga amb més unitats: {top_botiga[0]} ({top_botiga[1]} unitats)")
print(f"Article amb més unitats: {top_article[0]} ({top_article[1]} unitats)")
print(f"Incidència de rebuig més freqüent: {motiu_principal[0]} ({motiu_principal[1]} casos)")
print("Conclusió: aquestes agrupacions permeten comparar les unitats venudes entre famílies, botigues i articles.")


Registres vàlids: 84 de 120
Percentatge de registres vàlids: 70.0%
Total d'unitats: 3387
Família amb més unitats: Pilotes (818 unitats)
Botiga amb més unitats: Girona (892 unitats)
Article amb més unitats: Raqueta (638 unitats)
Incidència de rebuig més freqüent: Data amb format incorrecte o no vàlida (19 casos)
Conclusió: aquestes agrupacions permeten comparar les unitats venudes entre famílies, botigues i articles.
